# OCR Pipeline แบบแยกโหลดโมเดล

Notebook นี้ออกแบบให้รัน TyphoonOCR หรือ Qwen/VLM ได้ทีละตัวผ่าน `MODEL_MODE` เพื่อเลี่ยงปัญหา dependency ของ `torch`, `transformers` หรือ model runtime ตีกันใน environment เดียวกัน

โหมดที่รองรับ:
- `MODEL_MODE = "typhoon_ocr"`: ใช้ `typhoon-ai/typhoon-ocr1.5-2b` อ่าน OCR จากรูปภาพ
- `MODEL_MODE = "qwen_vl"`: ใช้ `Qwen/Qwen3.5-2B` หรือ Qwen-VL 3.0 อ่าน/ตรวจ/จัดรูปแบบ/สรุปจากรูปภาพ หรือใช้ raw OCR เดิมถ้ามี

อ่านเฉพาะไฟล์รูปภาพจาก `renders/` เท่านั้น และ export CSV ไปที่ `ocr_submission.csv`

## Install Dependencies

ให้ติดตั้งเฉพาะ environment ที่จะใช้จริง เพื่อลดโอกาส dependency ตีกัน

In [ ]:
# Typhoon OCR environment. Uncomment and run only in the Typhoon env.
# %pip install -U "transformers>=4.57.0" accelerate pillow pandas tqdm torch torchvision sentencepiece protobuf

# Qwen/Qwen-VL environment. Uncomment and run only in the Qwen env.
# %pip install -U "transformers>=4.57.0" accelerate pillow pandas tqdm torch torchvision sentencepiece protobuf qwen-vl-utils

## Setup และ Config

เลือกโมเดลที่จะโหลดด้วย `MODEL_MODE` โดย notebook จะไม่โหลดอีกโมเดลหนึ่งเลย

In [1]:
from __future__ import annotations

import gc
import json
import re
import traceback
from pathlib import Path
from typing import Any

import pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

# Choose exactly one model/runtime per run.
MODEL_MODE = "typhoon_ocr"  # options: "typhoon_ocr", "qwen_vl"

# Model IDs. Change QWEN_MODEL_ID to Qwen/Qwen3-VL-8B-Instruct if you want Qwen-VL 3.0 instead of the 2B model.
TYPHOON_MODEL_ID = "typhoon-ai/typhoon-ocr1.5-2b"
QWEN_MODEL_ID = "Qwen/Qwen3.5-2B"
# QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# Paths
PROJECT_ROOT = Path("/Users/moi/PROJECT/Fah-Mai/fah-mai-the-finale-enterprise-data-agentic-showdown")
RENDERS_DIR = PROJECT_ROOT / "renders"
SAMPLE_SUBMISSION_PATH = PROJECT_ROOT / "sample__submission.csv"
OUTPUT_SUBMISSION_PATH = PROJECT_ROOT / "ocr_submission.csv"
INTERMEDIATE_JSON_PATH = PROJECT_ROOT / "ocr_intermediate_results.json"

# Input filtering. Do not add PDF/DOCX/TXT here; this notebook is image-only.
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}

# Runtime knobs
LIMIT_IMAGES = None  # set to 10 for a quick smoke test
LOAD_PREVIOUS_OCR_IN_QWEN_MODE = True  # lets Qwen clean Typhoon output from a previous run/env
SAVE_INTERMEDIATE_JSON = True
MAX_NEW_TOKENS = 4096

# macOS-friendly defaults. Use "mps" on Apple Silicon if available, otherwise CPU.
# Set DEVICE = "cpu" if MPS causes unsupported-op errors.
DEVICE = "auto"

if MODEL_MODE not in {"typhoon_ocr", "qwen_vl"}:
    raise ValueError('MODEL_MODE must be either "typhoon_ocr" or "qwen_vl"')

print(f"Project root      : {PROJECT_ROOT}")
print(f"Renders directory : {RENDERS_DIR}")
print(f"Model mode        : {MODEL_MODE}")
print(f"Output CSV        : {OUTPUT_SUBMISSION_PATH}")

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Project root      : /Users/moi/PROJECT/Fah-Mai/fah-mai-the-finale-enterprise-data-agentic-showdown
Renders directory : /Users/moi/PROJECT/Fah-Mai/fah-mai-the-finale-enterprise-data-agentic-showdown/renders
Model mode        : typhoon_ocr
Output CSV        : /Users/moi/PROJECT/Fah-Mai/fah-mai-the-finale-enterprise-data-agentic-showdown/ocr_submission.csv


## Load Images

สแกนเฉพาะ image files จาก `renders/` แบบ recursive และ ignore ไฟล์อื่นทั้งหมด

In [2]:
def derive_artifact_id(image_path: Path) -> str:
    """Map image filename back to an artifact id used by sample__submission.csv."""
    stem = image_path.stem

    # Bank statements are rendered as multiple images per artifact.
    stem = re.sub(r"_transactions_p\d+$", "", stem)
    stem = re.sub(r"_header$", "", stem)

    return stem


def image_sort_key(image_path: Path) -> tuple[str, int, str]:
    """Keep pages from the same artifact together and in readable order."""
    artifact_id = derive_artifact_id(image_path)
    stem = image_path.stem
    if stem.endswith("_header"):
        page_no = 0
    else:
        match = re.search(r"_transactions_p(\d+)$", stem)
        page_no = int(match.group(1)) if match else 1
    return (artifact_id, page_no, image_path.as_posix())


if not RENDERS_DIR.exists():
    raise FileNotFoundError(
        f"ไม่พบโฟลเดอร์ renders: {RENDERS_DIR}\n"
        "กรุณาตรวจ path หรือรัน notebook ในเครื่องที่มี data bundle"
    )

all_files = [path for path in RENDERS_DIR.rglob("*") if path.is_file()]
image_paths = sorted(
    [path for path in all_files if path.suffix.lower() in IMAGE_EXTENSIONS],
    key=image_sort_key,
)
ignored_files = [path for path in all_files if path.suffix.lower() not in IMAGE_EXTENSIONS]

if LIMIT_IMAGES is not None:
    image_paths = image_paths[:LIMIT_IMAGES]

if not image_paths:
    raise RuntimeError(
        "ไม่พบไฟล์รูปภาพใน renders/\n"
        "Notebook นี้รองรับเฉพาะ .png, .jpg, .jpeg, .webp และจะไม่ process PDF, DOCX, TXT หรือไฟล์อื่น"
    )

image_index = pd.DataFrame(
    [
        {
            "artifact_id": derive_artifact_id(path),
            "filename": path.name,
            "file_path": path.relative_to(PROJECT_ROOT).as_posix(),
            "folder": path.parent.relative_to(RENDERS_DIR).as_posix(),
        }
        for path in image_paths
    ]
)

print(f"Found image files : {len(image_paths):,}")
print(f"Ignored non-images: {len(ignored_files):,}")
print(f"Unique artifacts  : {image_index['artifact_id'].nunique():,}")
display(image_index.head(10))

Found image files : 6,047
Ignored non-images: 84
Unique artifacts  : 3,669


,artifact_id,filename,file_path,folder
0,BN-MEGA1111-2567,BN-MEGA1111-2567.png,renders/e7_banner/2024-11/BN-MEGA1111-2567.png,e7_banner/2024-11
1,BN-MEGA1111-2568,BN-MEGA1111-2568.png,renders/e7_banner/2025-11/BN-MEGA1111-2568.png,e7_banner/2025-11
2,BN-MEGA1212-2567,BN-MEGA1212-2567.png,renders/e7_banner/2024-12/BN-MEGA1212-2567.png,e7_banner/2024-12
3,BN-MEGA1212-2568,BN-MEGA1212-2568.png,renders/e7_banner/2025-12/BN-MEGA1212-2568.png,e7_banner/2025-12
4,BS-BBL-OPER-2567-01,BS-BBL-OPER-2567-01_header.png,renders/bank_statement/2024-01/BS-BBL-OPER-256...,bank_statement/2024-01
5,BS-BBL-OPER-2567-01,BS-BBL-OPER-2567-01_transactions_p1.png,renders/bank_statement/2024-01/BS-BBL-OPER-256...,bank_statement/2024-01
6,BS-BBL-OPER-2567-02,BS-BBL-OPER-2567-02_header.png,renders/bank_statement/2024-02/BS-BBL-OPER-256...,bank_statement/2024-02
7,BS-BBL-OPER-2567-02,BS-BBL-OPER-2567-02_transactions_p1.png,renders/bank_statement/2024-02/BS-BBL-OPER-256...,bank_statement/2024-02
8,BS-BBL-OPER-2567-03,BS-BBL-OPER-2567-03_header.png,renders/bank_statement/2024-03/BS-BBL-OPER-256...,bank_statement/2024-03
9,BS-BBL-OPER-2567-03,BS-BBL-OPER-2567-03_transactions_p1.png,renders/bank_statement/2024-03/BS-BBL-OPER-256...,bank_statement/2024-03


## Load Sample Submission Schema

ถ้ามี `sample__submission.csv` จะใช้ column จากไฟล์นั้นเป็นหลัก แล้วเติม OCR result ให้ตรง format

In [3]:
if SAMPLE_SUBMISSION_PATH.exists():
    sample_df = pd.read_csv(SAMPLE_SUBMISSION_PATH, dtype=str).fillna("")
    sample_columns = list(sample_df.columns)
    print(f"Loaded sample submission: {SAMPLE_SUBMISSION_PATH}")
    print(f"Rows   : {len(sample_df):,}")
    print(f"Columns: {sample_columns}")
    display(sample_df.head(3))
else:
    sample_df = None
    sample_columns = ["id", "filename", "ocr_text", "cleaned_text"]
    print("ไม่พบ sample__submission.csv จะสร้าง submission schema พื้นฐานแทน")

Loaded sample submission: /Users/moi/PROJECT/Fah-Mai/fah-mai-the-finale-enterprise-data-agentic-showdown/sample__submission.csv
Rows   : 3,750
Columns: ['artifact_id', 'pred_json']


,artifact_id,pred_json
0,VI-V-013-INV-2567-226313,"{""payment_id"": ""BT-20240p-340727329532"", ""vend..."
1,WC-SKU-MASS-046-202405-10011040,"{""claim_id"": ""WC-2567-05-100110f0"", ""business_..."
2,BS-OPER-CNX-MAYA-2567-11,"{""L0_account_id"": ""OPER-CNX-MAYA"", ""L0_bank"": ..."


## Model Helpers

ส่วนนี้ import `torch` และ `transformers` แบบ lazy เฉพาะตอนโหลดโมเดลที่เลือก

In [4]:
TYPHOON_PROMPT = """Extract all text from this image.

Instructions:
- Return only readable Markdown.
- Keep all visible text, numbers, dates, IDs, amounts, and table content.
- Do not add explanations.
- If a value is unclear, preserve your best reading and mark it with [?].
"""

QWEN_VL_PROMPT_TEMPLATE = """You are a careful Thai/English OCR reviewer for business documents.

Read the image and use the raw OCR text if provided. Correct OCR mistakes, organize the content, identify important fields/tables, and summarize the image.
Do not invent values that are not visible in the image or present in the OCR text.

Return JSON only with this schema:
{{
  "ocr_text": "complete text read from the image",
  "cleaned_text": "corrected and organized Markdown",
  "structured_result": {{
    "document_type": "vendor_invoice | warranty_form | bank_statement | receipt | banner | lease_or_training_doc | unknown",
    "important_fields": {{}},
    "tables": []
  }},
  "summary": "short Thai summary",
  "confidence_notes": "short note about uncertainty or image quality"
}}

File: {filename}
Artifact ID: {artifact_id}

Previous raw OCR text, if any:
```text
{raw_ocr_text}
```
"""


def load_image(path: Path) -> Image.Image:
    """Open image as RGB. Corrupt files raise a clear error that the loop can catch."""
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as exc:
        raise RuntimeError(f"Cannot open image: {path}") from exc


def resize_image(image: Image.Image, long_side: int = 1800) -> Image.Image:
    """Resize large images to a practical size for local inference."""
    width, height = image.size
    current_long_side = max(width, height)
    if current_long_side <= long_side:
        return image
    scale = long_side / current_long_side
    new_size = (max(1, int(width * scale)), max(1, int(height * scale)))
    return image.resize(new_size, Image.Resampling.LANCZOS)


def resolve_device(torch_module: Any, requested: str = "auto") -> str:
    if requested != "auto":
        return requested
    if torch_module.cuda.is_available():
        return "cuda"
    if hasattr(torch_module.backends, "mps") and torch_module.backends.mps.is_available():
        return "mps"
    return "cpu"


def load_selected_model():
    """Load only the selected model. This is the key to keeping environments separate."""
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor

    model_id = TYPHOON_MODEL_ID if MODEL_MODE == "typhoon_ocr" else QWEN_MODEL_ID
    device = resolve_device(torch, DEVICE)
    print(f"Loading model: {model_id}")
    print(f"Device       : {device}")

    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

    load_kwargs = {"trust_remote_code": True}
    try:
        load_kwargs["torch_dtype"] = "auto"
        model = AutoModelForImageTextToText.from_pretrained(model_id, **load_kwargs)
    except TypeError:
        load_kwargs.pop("torch_dtype", None)
        load_kwargs["dtype"] = "auto"
        model = AutoModelForImageTextToText.from_pretrained(model_id, **load_kwargs)

    if device in {"cuda", "mps", "cpu"}:
        model = model.to(device)
    model.eval()
    return processor, model, torch, device


def generate_from_image(processor: Any, model: Any, torch_module: Any, device: str, image: Image.Image, prompt: str) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    if hasattr(inputs, "to"):
        inputs = inputs.to(device)

    with torch_module.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    input_len = inputs["input_ids"].shape[-1]
    generated_ids_trimmed = [output_ids[input_len:] for output_ids in generated_ids]
    return processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


def parse_json_or_text(text: str) -> dict[str, Any]:
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
    return {
        "ocr_text": cleaned,
        "cleaned_text": cleaned,
        "structured_result": {},
        "summary": "",
        "confidence_notes": "Model did not return valid JSON.",
    }


def free_memory(torch_module: Any | None = None):
    gc.collect()
    if torch_module is not None and torch_module.cuda.is_available():
        torch_module.cuda.empty_cache()

## Optional: Load Previous OCR

ใช้เมื่อรัน Typhoon ในอีก environment แล้วต้องการให้ Qwen ช่วย clean จากผลเดิม โดยไม่ต้องโหลด Typhoon ใน environment นี้

In [5]:
previous_by_file: dict[str, dict[str, Any]] = {}

if MODEL_MODE == "qwen_vl" and LOAD_PREVIOUS_OCR_IN_QWEN_MODE and INTERMEDIATE_JSON_PATH.exists():
    previous_records = json.loads(INTERMEDIATE_JSON_PATH.read_text(encoding="utf-8"))
    previous_by_file = {item.get("file_path", ""): item for item in previous_records}
    print(f"Loaded previous OCR records: {len(previous_by_file):,} from {INTERMEDIATE_JSON_PATH}")
else:
    print("No previous OCR records loaded.")

No previous OCR records loaded.


## Load Selected Model

Cell นี้จะโหลดเฉพาะโมเดลตาม `MODEL_MODE` เท่านั้น

In [ ]:
processor, model, torch, runtime_device = load_selected_model()

Loading model: typhoon-ai/typhoon-ocr1.5-2b
Device       : mps


/opt/anaconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 4255.14 MB. The target location /Users/moi/.cache/huggingface/hub/models--typhoon-ai--typhoon-ocr1.5-2b/blobs only has 1205.74 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

## Process All Images

วนอ่านภาพทีละไฟล์ พร้อม error handling รายภาพ ถ้าภาพเสียหรือ inference fail จะบันทึก error แล้วไปต่อ

In [ ]:
def process_one_image(image_path: Path) -> dict[str, Any]:
    rel_path = image_path.relative_to(PROJECT_ROOT).as_posix()
    artifact_id = derive_artifact_id(image_path)
    record = {
        "artifact_id": artifact_id,
        "filename": image_path.name,
        "file_path": rel_path,
        "model_mode": MODEL_MODE,
        "ocr_text": "",
        "cleaned_text": "",
        "structured_result": {},
        "summary": "",
        "confidence_notes": "",
        "status": "pending",
        "error": "",
    }

    try:
        image = resize_image(load_image(image_path))

        if MODEL_MODE == "typhoon_ocr":
            text = generate_from_image(processor, model, torch, runtime_device, image, TYPHOON_PROMPT)
            record["ocr_text"] = text
            record["cleaned_text"] = text
            record["structured_result"] = {
                "document_type": image_path.parts[-3] if len(image_path.parts) >= 3 else "unknown",
                "source": "typhoon_ocr_raw",
            }
            record["summary"] = ""
            record["status"] = "done"

        elif MODEL_MODE == "qwen_vl":
            previous = previous_by_file.get(rel_path, {})
            raw_ocr_text = previous.get("ocr_text") or previous.get("raw_ocr_text") or ""
            prompt = QWEN_VL_PROMPT_TEMPLATE.format(
                filename=image_path.name,
                artifact_id=artifact_id,
                raw_ocr_text=raw_ocr_text,
            )
            text = generate_from_image(processor, model, torch, runtime_device, image, prompt)
            parsed = parse_json_or_text(text)
            record["ocr_text"] = parsed.get("ocr_text") or raw_ocr_text or ""
            record["cleaned_text"] = parsed.get("cleaned_text") or record["ocr_text"]
            record["structured_result"] = parsed.get("structured_result", {})
            record["summary"] = parsed.get("summary", "")
            record["confidence_notes"] = parsed.get("confidence_notes", "")
            record["qwen_raw_output"] = text
            record["status"] = "done"

    except Exception as exc:
        record["status"] = "failed"
        record["error"] = f"{type(exc).__name__}: {exc}"
        record["traceback"] = traceback.format_exc(limit=3)

    return record


records: list[dict[str, Any]] = []

for image_path in tqdm(image_paths, desc=f"Processing images with {MODEL_MODE}"):
    print(f"Processing: {image_path.relative_to(PROJECT_ROOT)}")
    records.append(process_one_image(image_path))

results_df = pd.DataFrame(records)
print("Finished processing images.")
display(results_df.head(10))
display(results_df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))

## Save Intermediate Results

ไฟล์นี้ช่วยให้รัน Typhoon env ก่อน แล้วเปิดอีก env เพื่อให้ Qwen clean ต่อได้ โดยไม่ต้องโหลดสองโมเดลพร้อมกัน

In [ ]:
if SAVE_INTERMEDIATE_JSON:
    INTERMEDIATE_JSON_PATH.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved intermediate results: {INTERMEDIATE_JSON_PATH}")

## Build Submission DataFrame

รวมผลหลายภาพกลับเป็น artifact เดียว แล้วสร้าง CSV ตาม schema ของ sample ถ้ามี

In [ ]:
def join_nonempty(values: list[str], separator: str = "\n\n---\n\n") -> str:
    return separator.join(value for value in values if isinstance(value, str) and value.strip())


def merge_structured_results(values: list[Any]) -> list[Any]:
    merged = []
    for value in values:
        if isinstance(value, dict) and value:
            merged.append(value)
        elif isinstance(value, list) and value:
            merged.extend(value)
    return merged


artifact_rows = []
for artifact_id, group in results_df.groupby("artifact_id", sort=False):
    group_records = group.to_dict(orient="records")
    source_files = [row["file_path"] for row in group_records]
    errors = [row.get("error", "") for row in group_records if row.get("error")]
    payload = {
        "ocr_text": join_nonempty([row.get("ocr_text", "") for row in group_records]),
        "cleaned_text": join_nonempty([row.get("cleaned_text", "") for row in group_records]),
        "summary": join_nonempty([row.get("summary", "") for row in group_records], separator="\n"),
        "structured_result": merge_structured_results([row.get("structured_result", {}) for row in group_records]),
        "source_files": source_files,
        "model_mode": MODEL_MODE,
        "errors": errors,
    }
    artifact_rows.append(
        {
            "artifact_id": artifact_id,
            "filename": source_files[0] if source_files else "",
            "ocr_text": payload["ocr_text"],
            "cleaned_text": payload["cleaned_text"],
            "pred_json": json.dumps(payload, ensure_ascii=False),
        }
    )

artifact_df = pd.DataFrame(artifact_rows)
print(f"Artifact rows built: {len(artifact_df):,}")
display(artifact_df.head(5))

In [ ]:
if sample_df is not None:
    submission_df = sample_df.copy()

    if "artifact_id" not in submission_df.columns:
        raise ValueError("sample__submission.csv exists but has no artifact_id column")

    payload_by_artifact = artifact_df.set_index("artifact_id")["pred_json"].to_dict()
    if "pred_json" in submission_df.columns:
        submission_df["pred_json"] = submission_df["artifact_id"].map(payload_by_artifact).fillna("{}")
    else:
        submission_df["pred_json"] = submission_df["artifact_id"].map(payload_by_artifact).fillna("{}")

    # Preserve original sample columns first, then append pred_json if it was absent.
    output_columns = list(sample_columns)
    if "pred_json" not in output_columns:
        output_columns.append("pred_json")
    submission_df = submission_df[output_columns]

else:
    submission_df = artifact_df.rename(columns={"artifact_id": "id"})[["id", "filename", "ocr_text", "cleaned_text"]]

print(f"Submission rows: {len(submission_df):,}")
display(submission_df.head(10))

## Export CSV

บันทึกไฟล์สุดท้ายไปยัง path ที่กำหนด

In [ ]:
OUTPUT_SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(OUTPUT_SUBMISSION_PATH, index=False, encoding="utf-8-sig")
print(f"Saved submission CSV: {OUTPUT_SUBMISSION_PATH}")

## Preview Output File

ตรวจตัวอย่างไฟล์ที่ export แล้ว

In [ ]:
preview_df = pd.read_csv(OUTPUT_SUBMISSION_PATH, dtype=str).fillna("")
print(preview_df.shape)
display(preview_df.head(10))

## วิธีรันแนะนำ

1. รันด้วย `MODEL_MODE = "typhoon_ocr"` ใน environment สำหรับ Typhoon เพื่อสร้าง `ocr_intermediate_results.json` และ `ocr_submission.csv`
2. ถ้าต้องการให้ Qwen ตรวจ/clean เพิ่ม ให้เปิดอีก environment ที่รองรับ Qwen แล้วตั้ง `MODEL_MODE = "qwen_vl"`
3. ให้ `LOAD_PREVIOUS_OCR_IN_QWEN_MODE = True` เพื่อใช้ raw OCR จาก `ocr_intermediate_results.json`
4. รันต่อจน export `ocr_submission.csv` อีกครั้ง